# Exploratory multivariate gold-volatility forecast

This notebook tests whether the Stage 2 candidate predictors have conditional linear associations with five-session forward realized variance, then evaluates a nonlinear multivariate forecast. It uses the prepared Stage 2 dataset and its manifest; run `04_predictors.ipynb` first if those files are absent.

**Inference note:**  The coefficient tests below are training-only OLS t-tests with HAC/Newey–West covariance, allowing for heteroskedasticity and dependence from overlapping five-session labels. They are exploratory associations, not causal effects or a final rule for selecting variables. All seven candidate predictors remain in the forecast model regardless of these p-values.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.stats.multitest import multipletests

repo_root = Path.cwd()
if not (repo_root / 'pyproject.toml').is_file():
    repo_root = repo_root.parent
if not (repo_root / 'pyproject.toml').is_file():
    raise FileNotFoundError('Could not locate repository root (pyproject.toml).')
sys.path.insert(0, str(repo_root / 'src'))

data_path = repo_root / 'data/processed/stage2_multivariate.csv'
manifest_path = repo_root / 'data/processed/stage2_multivariate_manifest.json'
if not data_path.is_file() or not manifest_path.is_file():
    raise FileNotFoundError(
        'Stage 2 data are missing. Run notebooks/04_predictors.ipynb first.'
    )

data = pd.read_csv(data_path, parse_dates=['date'])
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
features = [
    'dxy_close',
    'sp500_log_return',
    'vix_close',
    'treasury_10y_yield',
    'breakeven_inflation_10y',
    'wti_spot',
    'gold_volume',
]
target = manifest.get('target', 'realized_variance')
horizon = int(manifest.get('horizon', 5))
required = ['date', 'log_return', 'target_end_date', target, *features]
missing = sorted(set(required) - set(data.columns))
if missing:
    raise ValueError(f'Prepared dataset is missing required columns: {missing}')

data = data.sort_values('date').reset_index(drop=True)
analysis = data[required].replace([np.inf, -np.inf], np.nan)
analysis['target_end_date'] = pd.to_datetime(analysis['target_end_date'], errors='raise')
missing_counts = analysis[features + [target]].isna().sum()
if missing_counts.any():
    display(missing_counts[missing_counts > 0].rename('missing_rows').to_frame())
analysis = analysis.dropna(subset=features + [target]).reset_index(drop=True)
split_at = int(len(analysis) * 0.8)
if split_at < 30 or len(analysis) - split_at < 10:
    raise ValueError('Not enough complete observations for the chronological split.')

test = analysis.iloc[split_at:].copy()
test_start = test['date'].min()
train = analysis.iloc[:split_at].loc[
    analysis.iloc[:split_at]['target_end_date'] <= test_start
].copy()
if len(train) < 30:
    raise ValueError('Not enough training observations after the target-window purge.')
print(f'Complete observations: {len(analysis):,}')
print(f'Train: {train.date.min().date()} to {train.date.max().date()} ({len(train):,})')
print(f'Holdout: {test.date.min().date()} to {test.date.max().date()} ({len(test):,})')
print(f'Purged training rows whose target ends after the first holdout origin: {split_at - len(train)}')
print(f'Predictors ({len(features)}): {", ".join(features)}')
print(f'Target: {horizon}-session forward {target}')

Complete observations: 2,976
Train: 2015-01-05 to 2024-06-21 (2,380)
Holdout: 2024-06-24 to 2026-11-03 (596)
Predictors (7): dxy_close, sp500_log_return, vix_close, treasury_10y_yield, breakeven_inflation_10y, wti_spot, gold_volume
Target: 5-session forward realized_variance


## Predictor significance screen (training period only)

Before fitting, training labels are purged so every retained row has `target_end_date` on or before the first holdout origin. Standardize predictors using training-period means and standard deviations, then regress `log1p(realized_variance)` on all candidate predictors. Report HAC-robust coefficient t-statistics with `horizon - 1` lags to account for overlapping labels, plus Holm-adjusted p-values for the seven coefficient tests. The holdout period is not used for these tests.

In [2]:
train_means = train[features].mean()
train_scales = train[features].std(ddof=0).replace(0, 1)
X_train_standardized = (train[features] - train_means) / train_scales
y_train_log = np.log1p(train[target])
hac_lags = max(horizon - 1, 0)

significance_model = sm.OLS(
    y_train_log,
    sm.add_constant(X_train_standardized, has_constant='add'),
).fit(
    cov_type='HAC',
    cov_kwds={'maxlags': hac_lags},
    use_t=True,
)
coefficient_names = features
p_values = significance_model.pvalues[coefficient_names]
holm_p_values = multipletests(p_values, method='holm')[1]
significance = pd.DataFrame(
    {
        'predictor': coefficient_names,
        'coefficient_per_train_sd': significance_model.params[coefficient_names].to_numpy(),
        'HAC_t_statistic': significance_model.tvalues[coefficient_names].to_numpy(),
        'HAC_p_value': p_values.to_numpy(),
        'Holm_adjusted_p_value': holm_p_values,
    }
).sort_values('Holm_adjusted_p_value')
display(significance.style.format({
    'coefficient_per_train_sd': '{:.4g}',
    'HAC_t_statistic': '{:.3f}',
    'HAC_p_value': '{:.4g}',
    'Holm_adjusted_p_value': '{:.4g}',
}).hide(axis='index'))
print(f'Training R-squared: {significance_model.rsquared:.3f}')
print(f'HAC maximum lag: {hac_lags}; multiplicity adjustment: Holm')
print('Predictors are not removed based on this exploratory significance screen.')

predictor,coefficient_per_train_sd,HAC_t_statistic,HAC_p_value,Holm_adjusted_p_value
vix_close,0.0002677,4.093,4.405e-05,0.0003084
breakeven_inflation_10y,-0.0001189,-2.015,0.04405,0.2643
dxy_close,-4.857e-06,-0.176,0.8604,1
sp500_log_return,-1.319e-05,-0.703,0.4822,1
treasury_10y_yield,2.821e-05,0.729,0.466,1
wti_spot,-1.139e-06,-0.024,0.9813,1
gold_volume,2.689e-05,0.951,0.3417,1


Training R-squared: 0.291
HAC maximum lag: 4; multiplicity adjustment: Holm
Predictors are not removed based on this exploratory significance screen.


## LSEG predictor significance screen

Repeat the training-only HAC screen on the separate LSEG `GCc1` dataset. The prepared panel uses a one-weekday availability proxy, not exact publication timestamps. Test the available 11 predictors (using the S&P 500 return rather than its level), apply Holm correction across individual coefficients, and use joint HAC tests to assess the correlated gold implied-volatility tenors as a group. This remains exploratory and does not establish out-of-sample predictive value.

In [ ]:
lseg_data_path = repo_root / 'data/processed/stage2_lseg_multivariate.csv'
lseg_manifest_path = repo_root / 'data/processed/stage2_lseg_multivariate_manifest.json'
if not lseg_data_path.is_file() or not lseg_manifest_path.is_file():
    raise FileNotFoundError(
        'LSEG data are missing. Run '
        'python -m gold_vol_pipeline.predictors --lseg-input-dir data/refinitiv first.'
    )

lseg_data = pd.read_csv(lseg_data_path, parse_dates=['date'])
lseg_manifest = json.loads(lseg_manifest_path.read_text(encoding='utf-8'))
lseg_features = [
    'dxy_close',
    'sp500_log_return',
    'vix_close',
    'treasury_10y_yield',
    'gold_iv_1m',
    'gold_iv_2m',
    'gold_iv_3m',
    'gold_iv_6m',
    'gold_iv_1y',
    'gold_1m_risk_reversal',
    'gold_1m_butterfly',
]
lseg_target = lseg_manifest.get('target', 'realized_variance')
lseg_horizon = int(lseg_manifest.get('horizon', 5))
lseg_required = ['date', 'target_end_date', lseg_target, *lseg_features]
lseg_missing = sorted(set(lseg_required) - set(lseg_data.columns))
if lseg_missing:
    raise ValueError(f'Prepared LSEG data are missing columns: {lseg_missing}')

lseg_data = lseg_data.sort_values('date').reset_index(drop=True)
lseg_analysis = lseg_data[lseg_required].replace([np.inf, -np.inf], np.nan)
lseg_analysis['target_end_date'] = pd.to_datetime(
    lseg_analysis['target_end_date'], errors='raise'
)
lseg_analysis = lseg_analysis.dropna(
    subset=lseg_features + [lseg_target]
).reset_index(drop=True)
lseg_split_at = int(len(lseg_analysis) * 0.8)
lseg_test = lseg_analysis.iloc[lseg_split_at:].copy()
lseg_test_start = lseg_test['date'].min()
lseg_train = lseg_analysis.iloc[:lseg_split_at].loc[
    lseg_analysis.iloc[:lseg_split_at]['target_end_date'] <= lseg_test_start
].copy()
if len(lseg_train) < max(30, len(lseg_features) + 2):
    raise ValueError('Not enough LSEG training observations after target-window purge.')

lseg_means = lseg_train[lseg_features].mean()
lseg_scales = lseg_train[lseg_features].std(ddof=0).replace(0, 1)
lseg_X_train = (lseg_train[lseg_features] - lseg_means) / lseg_scales
lseg_y_train = np.log1p(lseg_train[lseg_target])
lseg_hac_lags = max(lseg_horizon - 1, 0)
lseg_model = sm.OLS(
    lseg_y_train,
    sm.add_constant(lseg_X_train, has_constant='add'),
).fit(
    cov_type='HAC',
    cov_kwds={'maxlags': lseg_hac_lags},
    use_t=True,
)

lseg_p_values = lseg_model.pvalues[lseg_features]
lseg_holm_p_values = multipletests(lseg_p_values, method='holm')[1]
lseg_significance = pd.DataFrame({
    'predictor': lseg_features,
    'coefficient_per_train_sd': lseg_model.params[lseg_features].to_numpy(),
    'HAC_t_statistic': lseg_model.tvalues[lseg_features].to_numpy(),
    'HAC_p_value': lseg_p_values.to_numpy(),
    'Holm_adjusted_p_value': lseg_holm_p_values,
}).sort_values('Holm_adjusted_p_value')
display(lseg_significance.style.format({
    'coefficient_per_train_sd': '{:.4g}',
    'HAC_t_statistic': '{:.3f}',
    'HAC_p_value': '{:.4g}',
    'Holm_adjusted_p_value': '{:.4g}',
}).hide(axis='index'))

lseg_groups = {
    'All gold options': lseg_features[4:],
    'Gold implied-volatility tenors': lseg_features[4:9],
    'Risk reversal and butterfly': lseg_features[9:],
}
lseg_group_results = []
for group_name, group_features in lseg_groups.items():
    restrictions = np.zeros((len(group_features), len(lseg_model.params)))
    for row, feature in enumerate(group_features):
        restrictions[row, lseg_model.params.index.get_loc(feature)] = 1
    group_test = lseg_model.f_test(restrictions)
    lseg_group_results.append({
        'group': group_name,
        'HAC_F_statistic': float(np.asarray(group_test.fvalue).squeeze()),
        'HAC_p_value': float(np.asarray(group_test.pvalue).squeeze()),
    })
lseg_group_results = pd.DataFrame(lseg_group_results)
lseg_group_results['Holm_adjusted_p_value'] = multipletests(
    lseg_group_results['HAC_p_value'], method='holm'
)[1]
display(lseg_group_results.style.format({
    'HAC_F_statistic': '{:.3f}',
    'HAC_p_value': '{:.4g}',
    'Holm_adjusted_p_value': '{:.4g}',
}).hide(axis='index'))

lseg_iv_features = lseg_features[4:9]
lseg_iv_correlations = lseg_train[lseg_iv_features].corr().abs()
lseg_max_iv_correlation = lseg_iv_correlations.where(
    ~np.eye(len(lseg_iv_features), dtype=bool)
).max().max()
print(
    f'LSEG complete rows: {len(lseg_analysis):,}; '
    f'train: {len(lseg_train):,}; holdout: {len(lseg_test):,}; '
    f'purged: {lseg_split_at - len(lseg_train):,}'
)
print(
    f'Train: {lseg_train.date.min().date()} to {lseg_train.date.max().date()}; '
    f'holdout: {lseg_test.date.min().date()} to {lseg_test.date.max().date()}'
)
print(f'Maximum absolute correlation across IV tenors: {lseg_max_iv_correlation:.3f}')
print(
    f'HAC maximum lag: {lseg_hac_lags}; Holm correction across '
    f'{len(lseg_features)} coefficients and {len(lseg_groups)} group tests.'
)

## Nonlinear multivariate forecast

Fit a histogram-based gradient boosting regressor to the log-transformed target using every candidate predictor. The split is chronological; no holdout observations are used to fit or tune the model. Compare against a training-mean forecast and an expanding historical-variance forecast based only on gold returns known at each origin. This single holdout is exploratory, not a standardized final model-selection exercise.

In [3]:
from gold_vol_pipeline.models import expanding_variance_forecast

X_train = train[features]
X_test = test[features]
y_train = train[target].to_numpy(dtype=float)
y_test = test[target].to_numpy(dtype=float)

model = HistGradientBoostingRegressor(
    loss='squared_error',
    learning_rate=0.05,
    max_iter=200,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    random_state=42,
)
model.fit(X_train, np.log1p(y_train))
model_forecast = np.maximum(np.expm1(model.predict(X_test)), 0.0)
mean_forecast = np.full(len(test), y_train.mean())

expanding_forecast = []
for origin_date in test['date']:
    origin_returns = data.loc[data['date'] <= origin_date, 'log_return'].dropna().to_numpy(dtype=float)
    expanding_forecast.append(
        expanding_variance_forecast(origin_returns, horizon=horizon)
    )
expanding_forecast = np.asarray(expanding_forecast)

forecasts = {
    'HistGradientBoosting (all predictors)': model_forecast,
    'Training-mean benchmark': mean_forecast,
    'Expanding-variance benchmark': expanding_forecast,
}
metrics = pd.DataFrame(
    [
        {
            'model': name,
            'MAE': mean_absolute_error(y_test, prediction),
            'RMSE': np.sqrt(mean_squared_error(y_test, prediction)),
            'R2': r2_score(y_test, prediction),
        }
        for name, prediction in forecasts.items()
    ]
).sort_values('RMSE')
display(metrics.style.format({'MAE': '{:.6g}', 'RMSE': '{:.6g}', 'R2': '{:.3f}'}).hide(axis='index'))

holdout_predictions = pd.DataFrame({
    'date': test['date'].to_numpy(),
    'actual': y_test,
    'hist_gradient_boosting': model_forecast,
    'training_mean': mean_forecast,
    'expanding_variance': expanding_forecast,
})
display(holdout_predictions.head())

model,MAE,RMSE,R2
Expanding-variance benchmark,0.000761527,0.00183267,-0.120
Training-mean benchmark,0.000786579,0.0018552,-0.148
HistGradientBoosting (all predictors),0.000813263,0.00188303,-0.183


,date,actual,hist_gradient_boosting,training_mean,expanding_variance
0,2024-06-24,0.000194,0.000461,0.000431,0.000432
1,2024-06-25,0.000166,0.000617,0.000431,0.000432
2,2024-06-26,0.000343,0.000728,0.000431,0.000432
3,2024-06-27,0.000383,0.000774,0.000431,0.000432
4,2024-06-28,0.000587,0.000727,0.000431,0.000431


## Interpretation and limitations

- The HAC coefficient tests are conditional linear associations in the training sample, not paired t-tests, causal effects, or proof of out-of-sample predictive value. Holm adjustment limits false positives across the tested coefficients.
- The model includes all seven selected candidate inputs: DXY close, S&P 500 log return, VIX close, 10-year Treasury yield, 10-year breakeven inflation, WTI spot, and optional gold-futures volume. The retained S&P 500 close level is not used because the selected equity predictor is its close-to-close return.
- The separate LSEG screen uses the partial `GCc1` panel. The current exports do not include gold volume, breakeven inflation, or WTI, and spot/fixing requests failed. DXY and VIX are individually significant in the training screen; the five implied-volatility tenors are significant jointly but highly collinear, so individual tenor coefficients are not stably distinguished. Verify vendor units and the one-weekday availability proxy before interpreting these results.
- FRED uses current downloaded values and a one-weekday availability proxy, not historical vintages or series-specific release timestamps. Verify this timing before a strict real-time backtest.
- Five-session target windows overlap, so forecast errors are dependent. HAC inference addresses finite-lag dependence in the coefficient screen; the single holdout metrics are descriptive and have no paired model-significance test.
- Gold volume is provider-reported continuous-futures volume. Results depend on this source, the current data snapshot, the 80/20 split, and the chosen model settings. No model or variable-selection decision is finalized here.